In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [5]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/bhavanasisunilkumar/tokenizer-results/project_bpe_16k.pkl"
train_path = "/kaggle/input/datasets/bhavanasisunilkumar/tokenizer-results/train_tokens1 (2).npy"
val_path = "/kaggle/input/datasets/bhavanasisunilkumar/tokenizer-results/val_tokens1 (1).npy"
train2_path = "/kaggle/input/datasets/bhavanasisunilkumar/next-200/train_tokens2.npy"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    bpe = pickle.load(f)

# Load token arrays
train_tokens = np.load(train_path, mmap_mode="r")
val_tokens = np.load(val_path, mmap_mode="r")
train2_tokens = np.load(train2_path, mmap_mode = "r")

print("BPE vocab size:", len(bpe.itos))
print("Train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))
print("Train 2 tokens" , len(train2_tokens))

BPE vocab size: 7999
Train tokens: 199999998
Validation tokens: 5000000
Train 2 tokens 200000000


In [6]:
from typing import List
class TokenDataset(torch.utils.data.Dataset):
    """Fixed-length chunks of a tokenized corpus for next-token-prediction training."""

    def __init__(self, token_ids: List[int], seq_len: int):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return max(0, (len(self.ids) - 1) // self.seq_len)

    def __getitem__(self, i):
        start = i * self.seq_len
        x = self.ids[start : start + self.seq_len]
        y = self.ids[start + 1 : start + self.seq_len + 1]
        return x, y

In [7]:
BATCH_SIZE

16

In [8]:
#prepare data and val loader

train_dataset  = TokenDataset(train2_tokens, MAX_SEQ_LEN)
val_dataset = TokenDataset(val_tokens, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [9]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [10]:

"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, past_kv=None, use_cache=False,  position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            cos = self.rope_cos[ position_offset:position_offset + T]
            sin = self.rope_sin[position_offset:position_offset + T]
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)
            
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k],dim=2)
            v = torch.cat([past_v, v],dim=2)
        new_kv = (k, v) if use_cache else None

  

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attn_scores = q @ k.transpose(-2, -1) * self.attention_scale
        T_q = q.size(2)
        T_k = k.size(2)
        if past_kv is None:

            mask = self.causal_mask[:T_q, :T_k]

            attn_scores = attn_scores.masked_fill(
                ~mask,
                float("-inf")
            )
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        out = attn_weights @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out, new_kv


In [11]:

"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE



class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x,  past_kv=None, use_cache=False,  position_offset=0):
        attn_out, new_kv = self.attn(
                self.ln1(x),
                past_kv=past_kv,
                use_cache=use_cache,
                position_offset=position_offset
            )

        x = x + attn_out
    
        x = x + self.ffn(
            self.ln2(x)
        )
    
        return x, new_kv   # unchanged from Question 1


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor, past_kvs=None, use_cache=False) -> torch.Tensor:
        B, T = idx.shape
        x =self.tok_emb(idx)
        if past_kvs is None:
            position_offset = 0
        else:
            position_offset = past_kvs[0][0].size(2)
        if self.cfg.pos_emb_type == "sinusoidal":
          x = x + self.pos_emb[position_offset:position_offset + T]
        new_kvs = [] if use_cache else None
        for i, block in enumerate(self.blocks):
          past_kv = (
                     None
                        if past_kvs is None
                        else past_kvs[i]
                    )
          x, new_kv = block(
            x,
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset
        )

          if use_cache:
                new_kvs.append(new_kv)
        
        logits = self.head(self.ln_f(x))
        if use_cache:
          return logits, new_kvs
        return logits # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")



In [12]:
@torch.no_grad()
def perplexity_of_loader(model, loader, device="cpu"):
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for src, target in loader:
        src = src.to(device)
        target = target.to(device)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(src)

            loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target.reshape(-1),
                reduction="sum"
            )

        total_nll += loss.item()
        total_tokens += target.numel()

    if total_tokens == 0:
        return float("inf")

    avg_nll = total_nll / total_tokens

    return math.exp(avg_nll)

In [13]:
import time
def train(train_loader, val_loader, model , optimizer, train_steps = 1000, N = 100):
    
    criterion =  nn.CrossEntropyLoss()
    step = 0
    hist ={
          "train_loss":[],
          "held_out_ppl":[]
    
      }
    best_ppl = float("inf")
    best_state = None
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    start_time = time.time()
    while(step < train_steps):
        for src, target in train_loader:
          src, target = src.to(device), target.to(device)
          optimizer.zero_grad()
          with torch.autocast(device_type="cuda", dtype=torch.float16):
                output = model(src)
                loss = criterion(
                    output.reshape(-1, output.size(-1)),
                    target.reshape(-1)
                )
            
          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()
          step += 1
          hist["train_loss"].append(
                    (step, loss.item())
                )
          if step % N==0:
            elapsed = time.time() - start_time
            ppl = perplexity_of_loader(model, val_loader,  device=device)
            print(
                    f"Step {step}/{train_steps} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Val PPL: {ppl:.4f} | "
                    f"Time: {elapsed / 60:.2f} min"
                )
            hist["held_out_ppl"].append((step, ppl))
            if ppl < best_ppl:
                    best_ppl = ppl

                    best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                    
                    torch.save({
                        "step": step,
                        "model_state_dict": best_state,
                        "optimizer_state_dict": optimizer.state_dict(),
                        "best_ppl": best_ppl,
                    }, "mha_rope_batch16_best.pt")
                
            model.train()
          if step >= train_steps:
                break
    if best_state is not None:
      model.load_state_dict(best_state)
    total_time = time.time() - start_time
    print(f"\nTraining completed in: {total_time / 60:.2f} min")
    print(f"Best held-out PPL: {best_ppl:.4f}")
    return hist, best_ppl


In [14]:

@torch.no_grad()
def generate_tokens(model, input, target_words:int =128):
  input_ids = bpe.encode(input)
  input_ids = torch.tensor(input_ids, dtype = torch.long, device=device).unsqueeze(0)
  model.eval()
  generated=[]
  word_count = 0
  while word_count < target_words:
    context = input_ids[:, -MAX_SEQ_LEN:]
    logits = model(context)
    next_logits = logits[:,-1,:]
    next_token = next_logits.argmax(dim = -1)
    token_id = next_token.item()
    token_bytes = bpe.itos[token_id]
    if token_bytes.endswith(b"</w>"):
            word_count += 1
    generated.append(token_id)
    input_ids = torch.cat((input_ids, next_token.unsqueeze(0)), dim = 1)
  string = bpe.decode(generated)
  return string

In [15]:
cfg_sin = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model_sin = DecoderLM(cfg_sin)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model_sin = torch.nn.DataParallel(model_sin)

model_sin = model_sin.to(device)
optimizer_sin = torch.optim.AdamW(model_sin.parameters(), lr =3e-4, weight_decay = 0.1)

Using 2 GPUs


In [16]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model_sin))

total parameters :  67091456


In [17]:
checkpoint = torch.load(
    "/kaggle/input/datasets/bhavanasisunilkumar/mha-rope/mha_rope_batch16_best.pt",
    map_location=device,
    weights_only=False
)

model_sin.load_state_dict(checkpoint["model_state_dict"])

optimizer_sin.load_state_dict(checkpoint["optimizer_state_dict"])

print("Best PPL:", checkpoint["best_ppl"])
print("Checkpoint step:", checkpoint["step"])

Best PPL: 4.4559395388994245
Checkpoint step: 12400


In [18]:

print("mha rope training started")
hist_sin, ppl_sin = train(train_loader, val_loader, model_sin , optimizer_sin, train_steps = 12500, N = 100)

print("MHA rope PPL:", ppl_sin)

mha rope training started
Step 100/12500 | Loss: 2.0612 | Val PPL: 4.5532 | Time: 1.21 min
Step 200/12500 | Loss: 2.4133 | Val PPL: 4.5815 | Time: 3.89 min
Step 300/12500 | Loss: 1.9348 | Val PPL: 4.5920 | Time: 6.57 min
Step 400/12500 | Loss: 2.1225 | Val PPL: 4.5893 | Time: 9.25 min
Step 500/12500 | Loss: 2.1328 | Val PPL: 4.5913 | Time: 11.93 min
Step 600/12500 | Loss: 1.6547 | Val PPL: 4.5724 | Time: 14.61 min
Step 700/12500 | Loss: 2.1870 | Val PPL: 4.5651 | Time: 17.29 min
Step 800/12500 | Loss: 2.1065 | Val PPL: 4.5514 | Time: 19.97 min
Step 900/12500 | Loss: 2.1045 | Val PPL: 4.5265 | Time: 22.67 min
Step 1000/12500 | Loss: 2.0053 | Val PPL: 4.5123 | Time: 25.37 min
Step 1100/12500 | Loss: 2.3296 | Val PPL: 4.4936 | Time: 28.07 min
Step 1200/12500 | Loss: 1.9058 | Val PPL: 4.5138 | Time: 30.77 min
Step 1300/12500 | Loss: 2.1138 | Val PPL: 4.4805 | Time: 33.46 min
Step 1400/12500 | Loss: 2.2631 | Val PPL: 4.4824 | Time: 36.16 min
Step 1500/12500 | Loss: 2.0074 | Val PPL: 4.4588 

In [19]:
@torch.no_grad()
def generate_tokens(
    model,
    input,
    target_words=128,
    mode="greedy",
    temperature=0.8,
    top_k=50
):
    model.eval()

    # Encode prompt
    ids = bpe.encode(input)

    if len(ids) == 0:
        raise ValueError("Input produced zero BPE tokens.")

    input_ids = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # Limit prompt to context
    input_ids = input_ids[:, -MAX_SEQ_LEN:]

    generated = []
    word_count = 0

    # ---------------------------------------
    # FIRST PASS
    # ---------------------------------------
    logits, past_kvs = model(
        input_ids,
        use_cache=True
    )

    while word_count < target_words:
        # cache_len = past_kvs[0][0].size(2)

        # if cache_len >= MAX_SEQ_LEN:
        #     print(
        #         f"MAX CONTEXT REACHED: "
        #         f"{cache_len}, words={word_count}"
        #     )
        #     break
    
        # if logits.size(1) == 0:
        #     print(
        #         f"EMPTY LOGITS: "
        #         f"cache={cache_len}, words={word_count}"
        #     )
        #     break

        next_logits = logits[:, -1, :]
        # next_logits[:, 0] = float("-inf")  # <pad>
        # next_logits[:, 2] = float("-inf")  # <s>
        # next_logits[:, 3] = float("-inf")  # </s>

        # -----------------------------------
        # GREEDY
        # -----------------------------------
        if mode == "greedy":

            next_token = torch.argmax(
                next_logits,
                dim=-1,
                keepdim=True
            )

        # -----------------------------------
        # TEMPERATURE
        # -----------------------------------
        elif mode == "temperature":

            next_logits = next_logits / temperature

            probs = torch.softmax(
                next_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

        # -----------------------------------
        # TOP-K + TEMPERATURE
        # -----------------------------------
        elif mode == "top_k":

            next_logits = next_logits / temperature

            k = min(
                top_k,
                next_logits.size(-1)
            )

            top_values, top_indices = torch.topk(
                next_logits,
                k,
                dim=-1
            )

            filtered_logits = torch.full_like(
                next_logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                -1,
                top_indices,
                top_values
            )

            probs = torch.softmax(
                filtered_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

        else:
            raise ValueError(
                "mode must be 'greedy', "
                "'temperature', or 'top_k'"
            )

        # -----------------------------------
        # Count generated words
        # -----------------------------------

        token_id = next_token.item()

        token_bytes = bpe.itos[token_id]

        if token_bytes.endswith(b"</w>"):
            word_count += 1

        generated.append(token_id)

        # -----------------------------------
        # KV CACHE
        # -----------------------------------
        cache_len = past_kvs[0][0].size(2)
        
        if word_count >= target_words:
            break
        
        if cache_len < MAX_SEQ_LEN:
        
            logits, past_kvs = model(
                next_token,
                past_kvs=past_kvs,
                use_cache=True
            )

        else:
        
            generated_tensor = torch.tensor(
                generated,
                dtype=torch.long,
                device=device
            ).unsqueeze(0)
        
            context = torch.cat(
                [input_ids, generated_tensor],
                dim=1
            )
        
            context = context[:, -MAX_SEQ_LEN:]
        
            logits, past_kvs = model(
                context,
                use_cache=True
            )
        
            del generated_tensor
            del context

    return bpe.decode(generated)

In [20]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:")
print(test_file_path)
with open(test_file_path,"r", encoding="utf-8") as f:
    test_data = json.load(f)

test.json: 0.00B [00:00, ?B/s]

Test file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [21]:
predictions = {}

model_for_generation = (
    model_sin.module
    if isinstance(model_sin, torch.nn.DataParallel)
    else model_sin
)

for i, (id, data) in enumerate(test_data.items()):

    print(f"\n[{i+1}/{len(test_data)}] Testing ID: {id}")

    try:
        text = generate_tokens(
            model_for_generation,
            data["text"],
            target_words=128,
            mode="top_k",
            temperature=0.8,
          top_k=50,
        )

        predictions[id] = text.replace("<s>","").replace("</s>","").replace("<pad>","")

        print("SUCCESS")

    except Exception as e:
        print(f"FAILED ID: {id}")
        print(type(e).__name__, ":", e)
        raise


[1/1000] Testing ID: 1
SUCCESS

[2/1000] Testing ID: 2
SUCCESS

[3/1000] Testing ID: 3
SUCCESS

[4/1000] Testing ID: 4
SUCCESS

[5/1000] Testing ID: 5
SUCCESS

[6/1000] Testing ID: 6
SUCCESS

[7/1000] Testing ID: 7
SUCCESS

[8/1000] Testing ID: 8
SUCCESS

[9/1000] Testing ID: 9
SUCCESS

[10/1000] Testing ID: 10
SUCCESS

[11/1000] Testing ID: 11
SUCCESS

[12/1000] Testing ID: 12
SUCCESS

[13/1000] Testing ID: 13
SUCCESS

[14/1000] Testing ID: 14
SUCCESS

[15/1000] Testing ID: 15
SUCCESS

[16/1000] Testing ID: 16
SUCCESS

[17/1000] Testing ID: 17
SUCCESS

[18/1000] Testing ID: 18
SUCCESS

[19/1000] Testing ID: 19
SUCCESS

[20/1000] Testing ID: 20
SUCCESS

[21/1000] Testing ID: 21
SUCCESS

[22/1000] Testing ID: 22
SUCCESS

[23/1000] Testing ID: 23
SUCCESS

[24/1000] Testing ID: 24
SUCCESS

[25/1000] Testing ID: 25
SUCCESS

[26/1000] Testing ID: 26
SUCCESS

[27/1000] Testing ID: 27
SUCCESS

[28/1000] Testing ID: 28
SUCCESS

[29/1000] Testing ID: 29
SUCCESS

[30/1000] Testing ID: 30
SUCCES

In [22]:
import json

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            id: {"text": text}
            for id, text in predictions.items()
        },
        f,
        ensure_ascii=False,
        indent=2
    )

In [23]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip


In [24]:
torch.save({
    "model_state_dict": {
        k: v.detach().cpu().clone()
        for k, v in model_sin.state_dict().items()
    },
    "optimizer_state_dict": optimizer_sin.state_dict(),
    "history": hist_sin
}, "mha_rope_batch16_final.pt")